# 03 LLaMA3 解码器架构走读

## 简介

本笔记本介绍 LLaMA3 的解码器专用 (decoder-only) 架构，展示其相对于原始 Transformer 的五项关键改进：
RMSNorm、RoPE、SwiGLU、GQA 和 Pre-Norm。同时演示 KV Cache 的 prefill/decode 机制和文本生成。

## 1. 导入

In [ ]:
import torch
from models.llama3.config import LLaMA3Config
from models.llama3.model import LLaMA3Model, LLaMA3ForCausalLM, TransformerBlock
from models.common.attention import MultiHeadAttention, GroupedQueryAttention
from models.common.normalization import RMSNorm
from models.common.feedforward import SwiGLUFFN
from models.common.positional_encoding import RotaryPositionalEncoding

# 创建小型演示配置
config = LLaMA3Config(
    dim=128, n_heads=4, n_kv_heads=2, n_layers=2,
    ff_hidden_dim=344, max_seq_len=64
)
print(f"LLaMA3 配置: dim={config.dim}, heads={config.n_heads}, kv_heads={config.n_kv_heads}")
print(f"  head_dim={config.head_dim}, layers={config.n_layers}")

## 2. LLaMA3 五项关键改进

| 改进项 | 原始 Transformer | LLaMA3 | 优势 |
|---|---|---|---|
| **归一化** | LayerNorm (Post-Norm) | RMSNorm (Pre-Norm) | 更快, 训练更稳定 |
| **位置编码** | Sinusoidal PE (绝对) | RoPE (旋转位置编码) | 相对位置, 长度外推 |
| **注意力** | MHA (标准多头) | GQA (分组查询) | 减少 KV Cache |
| **激活函数** | ReLU/GELU | SiLU (SwiGLU) | 效果更好 |
| **归一化位置** | 子层之后 (Post-Norm) | 子层之前 (Pre-Norm) | 梯度流动更顺畅 |

## 3. 对比：LayerNorm vs RMSNorm

In [ ]:
from models.common.normalization import LayerNorm

x = torch.randn(2, 4, 64)
ln = LayerNorm(64)
rms = RMSNorm(64)

ln_out = ln(x)
rms_out = rms(x)

print(f"LayerNorm: {x.shape} -> {ln_out.shape}")
print(f"  γ 参数: {ln.weight.numel()}, β 参数: {ln.bias.numel()}")
print(f"  均值: {ln_out.mean(-1).detach().numpy()[0, :4]} (≈ 0)")
print(f"  标准差: {ln_out.std(-1).detach().numpy()[0, :4]} (≈ 1)")

print(f"\nRMSNorm: {x.shape} -> {rms_out.shape}")
print(f"  γ 参数: {rms.weight.numel()}, 无 β (比 LayerNorm 少 50% 参数)")
print(f"  RMS: {rms_out.norm(dim=-1).detach().numpy()[0, :4] / (64**0.5)}")

## 4. RoPE 旋转位置编码

RoPE 通过将 Q/K 向量在二维子空间上旋转来注入位置信息。
与正弦绝对位置编码不同，RoPE 天然编码相对位置关系。

In [ ]:
rope = RotaryPositionalEncoding(dim=32, max_seq_len=64, theta=10000.0)

q = torch.randn(1, 4, 16, 32)  # (B, n_heads, seq, head_dim)
k = torch.randn(1, 4, 16, 32)

q_rot, k_rot = rope(q, k)

print(f"RoPE 输入形状: q={list(q.shape)}, k={list(k.shape)}")
print(f"RoPE 输出形状: q={list(q_rot.shape)}, k={list(k_rot.shape)}")
print(f"\nRoPE 参数: max_seq_len={rope.max_seq_len}, theta={rope.theta}")
print(f"cos/sin 缓存形状: {list(rope.cos_cached.shape)}")

# 验证: RoPE 不影响向量模长
print(f"\n旋转前 Q 模长 (位置0): {q[0,0,0].norm():.4f}")
print(f"旋转后 Q 模长 (位置0): {q_rot[0,0,0].norm():.4f}")

## 5. SwiGLU 前馈网络

In [ ]:
from models.common.feedforward import FFN

dim = 128
ffn = FFN(dim=dim, hidden_dim=344)  # 标准: dim → hidden → dim
swiglu = SwiGLUFFN(dim=dim, hidden_dim=344)  # SwiGLU: 3 个投影

x = torch.randn(2, 8, dim)

print("标准 FFN vs SwiGLU FFN:")
print(f"  标准 FFN 投影: W1({dim}→344) + W2(344→{dim})")
print(f"  SwiGLU FFN 投影: W_gate({dim}→344) + W_up({dim}→344) + W_down(344→{dim})")

p_ffn = sum(p.numel() for p in ffn.parameters())
p_swiglu = sum(p.numel() for p in swiglu.parameters())
print(f"\n  标准 FFN 参数: {p_ffn:,}")
print(f"  SwiGLU FFN 参数: {p_swiglu:,} (+{(p_swiglu/p_ffn - 1)*100:.0f}%)")
print(f"\n  标准 FFN: {x.shape} -> {ffn(x).shape}")
print(f"  SwiGLU FFN: {x.shape} -> {swiglu(x).shape}")

## 6. TransformerBlock 走读

LLaMA3 的 TransformerBlock 使用 Pre-Norm 架构:

```
x = x + GQA(RMSNorm(x))   ← Self-Attention with RoPE
x = x + SwiGLU(RMSNorm(x)) ← Feed-Forward
```

In [ ]:
block = TransformerBlock(config)
x = torch.randn(2, 16, config.dim)

print("=" * 60)
print("TransformerBlock 逐步走读 (Pre-Norm)")
print("=" * 60)
print(f"输入: {list(x.shape)}")

# Pre-Norm: RMSNorm 在 attention 之前
x_norm = block.norm1(x)
q = block.attn.w_q(x_norm)
k = block.attn.w_k(x_norm)
v = block.attn.w_v(x_norm)
print(f"\n1. RMSNorm 后: {list(x_norm.shape)}")
print(f"2. Q 投影: {list(q.shape)}")
print(f"3. K 投影: {list(k.shape)} (n_kv_heads={config.n_kv_heads})")

q = block.attn._split_heads_q(q)
k = block.attn._split_heads_kv(k)
v = block.attn._split_heads_kv(v)
print(f"4. Q 分头: {list(q.shape)}")
print(f"5. K 分头: {list(k.shape)}")

# RoPE
q, k = block.rope(q, k)
print(f"6. RoPE 后 Q: {list(q.shape)}")

# Repeat KV
k = block.attn._repeat_kv(k)
v = block.attn._repeat_kv(v)
print(f"7. KV 扩展后 K: {list(k.shape)}")

# 最终输出
out = block(x)
print(f"\n最终输出: {list(out.shape)} (与输入一致: {out.shape == x.shape})")

## 7. KV Cache 演示：Prefill vs Decode

In [ ]:
# 构建模型
model = LLaMA3Model(config)

# 假设 prompt 长度为 4
prompt = torch.randint(0, config.vocab_size, (1, 4))
print(f"Prompt: {list(prompt.shape)}")

# === Prefill 阶段: 一次性处理整个 prompt ===
print("\n--- Prefill 阶段 ---")
kv_cache = model.create_kv_cache(batch_size=1)
h_prefill = model(prompt, kv_cache=kv_cache, start_pos=0)
print(f"Prefill 输出: {list(h_prefill.shape)}")
print(f"KV Cache 第0层 K 形状: {list(kv_cache[0][0].shape)}")
print(f"  已存储: 前 4 个位置的 K/V")

# === Decode 阶段: 逐 token 生成 ===
print("\n--- Decode 阶段 ---")
next_token = torch.randint(0, config.vocab_size, (1, 1))
print(f"新 token 输入: {list(next_token.shape)}")

h_decode = model(next_token, kv_cache=kv_cache, start_pos=4)
print(f"Decode 输出: {list(h_decode.shape)}")
print(f"KV Cache 第0层 K 形状: {list(kv_cache[0][0].shape)}")
print(f"  已存储: 前 5 个位置的 K/V (4个旧 + 1个新)")

print("\n★ Prefill 做一次 O(S²) 的注意力计算, 存入 KV Cache")
print("★ Decode 每步只做 O(S) 的注意力计算, 复用 KV Cache")

## 8. 文本生成 (Temperature 采样)

In [ ]:
lm = LLaMA3ForCausalLM(config)
lm.eval()

# 准备 prompt
prompt = torch.randint(0, config.vocab_size, (1, 4))

print("文本生成演示:")
print(f"  Prompt: {prompt.shape} — {prompt[0].tolist()}")

# 贪心生成 (temperature=0)
gen_greedy = lm.generate(prompt, max_new_tokens=8, temperature=0)
print(f"  贪心生成 (T=0): {gen_greedy.shape}")
print(f"    序列: {gen_greedy[0].tolist()}")

# 随机采样 (temperature=1.0)
gen_random = lm.generate(prompt, max_new_tokens=8, temperature=1.0)
print(f"  随机采样 (T=1.0): {gen_random.shape}")
print(f"    序列: {gen_random[0].tolist()}")

print("\n★ Temperature=0: 贪心解码，总是选概率最高的 token")
print("★ Temperature=1.0: 按 softmax 概率随机采样，更多样化")
print("★ Temperature>1.0: 分布更平坦，输出更随机")

## 9. 参数量对比: Transformer vs LLaMA3

In [ ]:
from models.transformer.config import TransformerConfig
from models.transformer.model import Transformer

# 可比较的配置 (相同 dim)
t_config = TransformerConfig(dim=256, n_heads=8, n_layers=8, ff_hidden_dim=1024, vocab_size=32000)
l_config = LLaMA3Config(dim=256, n_heads=8, n_kv_heads=4, n_layers=8, ff_hidden_dim=688, vocab_size=32000)

transformer = Transformer(t_config)
llama3 = LLaMA3ForCausalLM(l_config)

p_t = sum(p.numel() for p in transformer.parameters())
p_l = sum(p.numel() for p in llama3.parameters())

print(f"Transformer (Encoder+Decoder): {p_t:>12,} 参数")
print(f"LLaMA3 (Decoder only):        {p_l:>12,} 参数")
print(f"LLaMA3 参数占比: {p_l/p_t*100:.1f}%")

print("\nLLaMA3 虽然只有 Decoder, 但由于 SwiGLU 多了一个投影矩阵, ")
print("参数量大致为 Transformer Decoder 部分的约 1.5 倍")

## 11. 相关文档

本 notebook 对应的详细原理文档：

- [LLaMA 3 架构详解](../../docs/models/llama3.md) — RMSNorm、RoPE、SwiGLU、GQA、Pre-Norm 的完整原理和代码映射
- [Attention 机制详解](../../docs/models/attention.md) — GQA 的数学推导
- [KV Cache 原理](../../docs/models/llama3.md#kv-cache) — Prefill/Decode 阶段的 KV Cache 行为

建议阅读顺序：先运行本 notebook 建立直觉，再阅读文档理解数学细节。

## 11. 练习与思考

1. **RMSNorm 实现**: 从零实现 RMSNorm，验证输出与 PyTorch 内置版本一致
2. **RoPE 可视化**: 绘制不同位置的 Q 向量在二维子空间上的旋转效果，理解相对位置编码
3. **GQA 参数对比**: 固定 dim=4096，计算 n_kv_heads 分别为 1, 2, 4, 8, 32 时的参数量和 KV Cache 大小

## 10. 关键要点总结

1. **RMSNorm** 比 LayerNorm 少一半参数 (无 bias)，计算快约 10-15%
2. **RoPE** 将位置信息注入 Q/K 的旋转中，天然支持相对位置和长度外推
3. **SwiGLU** 使用 gate 机制，效果优于标准 FFN，但参数多 50%
4. **GQA** 用 2 或 4 个 KV 头替代 8 个，KV Cache 减少 2-4 倍
5. **Pre-Norm** 在子层前做归一化，训练更稳定，是 LLaMA 系列的默认选择
6. **KV Cache** 是推理加速的核心：Prefill 一次性计算并缓存 KV，Decode 复用缓存